In [2]:
import numpy as np
import math

class SimulatedPVSystem:
    """
    Simulates a PV Array under Partial Shading Conditions (PSC).
    This creates an environment with multiple local power peaks
    and one true Global Maximum Power Point (GMPP).
    """
    def evaluate_power(self, duty_cycle):
        # Clip duty cycle to practical hardware limits of a DC-DC converter
        d = np.clip(duty_cycle, 0.05, 0.95)

        # Superposition of multiple Gaussian functions representing localized shading peaks
        peak1 = 120 * np.exp(-((d - 0.25) / 0.1)**2)   # Local Peak 1
        peak2 = 250 * np.exp(-((d - 0.65) / 0.15)**2)  # Global Maximum Power Point (GMPP)
        peak3 = 90 * np.exp(-((d - 0.85) / 0.08)**2)   # Local Peak 2

        actual_power = peak1 + peak2 + peak3

        # Simulates low-frequency transducer noise from ADC sensors
        noise = np.random.normal(0, 0.05)
        return max(0.0, actual_power + noise)


class CuckooSearchMPPT:
    def __init__(self, population_size=6, pa=0.25, alpha=0.1):
        self.N = population_size
        self.pa = pa          # Fraction of worse nests discovered (alien eggs)
        self.alpha = alpha    # Step size scaling factor
        self.d_min = 0.05     # DC-DC converter lower boundary
        self.d_max = 0.95     # DC-DC converter upper boundary

        # Initialize internal arrays
        self.nests = np.random.uniform(self.d_min, self.d_max, self.N)
        self.fitness = np.zeros(self.N)

        self.best_d = None
        self.best_power = -float('inf')

    def levy_flight(self, Lambda=1.5):
        """
        Generates step sizes following a Levy Distribution using Mantegna's algorithm.
        Ensures proper handling using the standard python math framework.
        """
        # Calculate optimal step variance scaling factor (sigma)
        numerator = math.gamma(1 + Lambda) * np.sin(np.pi * Lambda / 2)
        denominator = math.gamma((1 + Lambda) / 2) * Lambda * 2**((Lambda - 1) / 2)
        sigma = (numerator / denominator)**(1 / Lambda)

        # Normal distribution variations
        u = np.random.normal(0, sigma, self.N)
        v = np.random.normal(0, 1, self.N)

        # Compute the heavy-tailed step array
        step = u / (np.abs(v)**(1 / Lambda))
        return step

    def execute_tracking(self, pv_system, max_iterations=25):
        # 1. Establish the baseline positions (First physical evaluations)
        for i in range(self.N):
            self.fitness[i] = pv_system.evaluate_power(self.nests[i])
            if self.fitness[i] > self.best_power:
                self.best_power = self.fitness[i]
                self.best_d = self.nests[i]

        print(f"Initial Scan complete. Starting Best Power: {self.best_power:.2f}W at Duty Cycle: {self.best_d:.4f}\n")

        # Main Optimization Iterative Control Loop
        for it in range(max_iterations):

            # Step A: Generate New Solutions using Levy Flights (Exploitation)
            steps = self.levy_flight()
            for i in range(self.N):
                # Calculate the step scaling vector comparative to global optimum position
                step_size = self.alpha * steps[i] * (self.nests[i] - self.best_d)
                new_d = np.clip(self.nests[i] + step_size, self.d_min, self.d_max)

                new_power = pv_system.evaluate_power(new_d)

                # Direct selection logic against a random host nest
                j = np.random.randint(0, self.N)
                if new_power > self.fitness[j]:
                    self.nests[j] = new_d
                    self.fitness[j] = new_power

            # Step B: Identify and Abandon Alien Eggs (Exploration)
            for i in range(self.N):
                if np.random.rand() < self.pa:
                    # Execute biased random walk variant using two random population elements
                    r1, r2 = np.random.choice(self.N, 2, replace=False)
                    step_size = np.random.rand() * (self.nests[r1] - self.nests[r2])
                    new_d = np.clip(self.nests[i] + step_size, self.d_min, self.d_max)

                    new_power = pv_system.evaluate_power(new_d)
                    if new_power > self.fitness[i]:
                        self.nests[i] = new_d
                        self.fitness[i] = new_power

            # Step C: Dynamic Global Memory Map Update
            for i in range(self.N):
                if self.fitness[i] > self.best_power:
                    self.best_power = self.fitness[i]
                    self.best_d = self.nests[i]

            # Periodic Status Outputs
            if (it + 1) % 5 == 0 or it == 0:
                print(f"Iteration {it+1:02d} -> Current Max Power: {self.best_power:.2f}W | Best Duty Cycle Command: {self.best_d:.4f}")

        return self.best_d, self.best_power


# --- Execution Execution ---
if __name__ == "__main__":
    # Fix the random state to ensure output tracking consistency
    np.random.seed(42)

    # Initialize physical plants and algorithmic tracker parameters
    pv_array_simulation = SimulatedPVSystem()
    controller = CuckooSearchMPPT(population_size=6, pa=0.25, alpha=0.1)

    print("==========================================================")
    print("      Cuckoo Search Global MPPT Controller Simulation     ")
    print("==========================================================\n")

    optimum_duty_cycle, max_extracted_power = controller.execute_tracking(
        pv_system=pv_array_simulation,
        max_iterations=25
    )

    print("\n==========================================================")
    print("                  Optimization Terminated                  ")
    print("==========================================================")
    print(f"Converged Target PWM Duty Cycle: {optimum_duty_cycle:.4f}")
    print(f"Max Attained Array Efficiency Power Output: {max_extracted_power:.2f} Watts")
    print("==========================================================")


      Cuckoo Search Global MPPT Controller Simulation     

Initial Scan complete. Starting Best Power: 218.36W at Duty Cycle: 0.7088

Iteration 01 -> Current Max Power: 225.16W | Best Duty Cycle Command: 0.6015
Iteration 05 -> Current Max Power: 246.98W | Best Duty Cycle Command: 0.6681
Iteration 10 -> Current Max Power: 246.98W | Best Duty Cycle Command: 0.6681
Iteration 15 -> Current Max Power: 247.00W | Best Duty Cycle Command: 0.6681
Iteration 20 -> Current Max Power: 247.00W | Best Duty Cycle Command: 0.6681
Iteration 25 -> Current Max Power: 247.03W | Best Duty Cycle Command: 0.6681

                  Optimization Terminated                  
Converged Target PWM Duty Cycle: 0.6681
Max Attained Array Efficiency Power Output: 247.03 Watts
